# Pipeline Multi-Ciudad con H3 — Carsharing Demand

Construye un dataset unificado de demanda de carsharing para las 10 ciudades europeas,
usando celdas H3 (resolución 8) como unidad espacial común.

**Salida:** `dataset_h3_multicidad.parquet`  
Columnas principales: `city`, `h3_cell`, `date`, `hour`, `day_of_week`, `month`,
`target_demanda`, `lag_1h`, `lag_24h`, `lag_168h`, `rolling_mean_3h`

**Ciudades:** amsterdam, berlin, firenze, kobenhavn, milano, muenchen, roma, stockholm, torino, wien  
**Resolución H3:** 8 (~0.7 km² por celda)

## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo si no están instaladas
# !pip install h3 pandas numpy pyarrow

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import h3
import re
from pathlib import Path
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# --- Rutas ---
DATA_DIR   = Path('../cs_datasets')          # carpeta con los *_trips.txt
OUTPUT_DIR = Path('.')                        # donde guardar el dataset final

# --- Parámetros H3 ---
H3_RESOLUTION = 8   # ~0.7 km² por celda; bajar a 7 (~5 km²) si hay pocas celdas activas

# --- Ciudades disponibles ---
CITIES = [
    'amsterdam', 'berlin', 'firenze', 'kobenhavn',
    'milano', 'muenchen', 'roma', 'stockholm', 'torino', 'wien'
]

print(f'h3 version: {h3.__version__}')
print(f'Ciudades a procesar: {len(CITIES)}')
print(f'DATA_DIR existe: {DATA_DIR.exists()}')

## 2. Funciones auxiliares

In [ ]:
# Formato de fecha en los txt:
# "Thu May 21 2015 23:32:48 GMT+0200 (W. Europe Daylight Time)"
_DATE_RE = re.compile(r'\s*\(.*?\)')

def parse_date(s: str) -> pd.Timestamp:
    """Parsea el formato de fecha de los trips.txt a Timestamp UTC."""
    clean = _DATE_RE.sub('', s).strip()          # quita el (timezone name)
    ts = pd.to_datetime(clean, format='%a %b %d %Y %H:%M:%S GMT%z')
    return ts.tz_convert('UTC')


def parse_coord(s: str):
    """Extrae (lat, lon) desde el formato 'lon,lat,0' de los trips.txt."""
    parts = s.strip().split(',')
    lon, lat = float(parts[0]), float(parts[1])
    return lat, lon


def coord_to_h3(lat: float, lon: float, resolution: int = H3_RESOLUTION) -> str:
    """Convierte coordenada a celda H3."""
    return h3.latlng_to_cell(lat, lon, resolution)


def load_city(city: str) -> pd.DataFrame:
    """
    Carga un archivo *_trips.txt, parsea fechas y coordenadas de origen,
    y devuelve un DataFrame con columnas:
        city, h3_cell, datetime_utc, date, hour, day_of_week, month
    """
    path = DATA_DIR / f'{city}_trips.txt'
    df = pd.read_csv(path, sep=' ', quotechar='"')

    # --- Parsear fechas de inicio ---
    print(f'  Parseando fechas ({city})...')
    df['datetime_utc'] = df['s_date'].apply(parse_date)

    # --- Parsear coordenadas de inicio ---
    coords = df['s_coord'].apply(parse_coord)
    df['lat'] = coords.apply(lambda x: x[0])
    df['lon'] = coords.apply(lambda x: x[1])

    # --- Asignar celda H3 ---
    print(f'  Asignando celdas H3 ({city})...')
    df['h3_cell'] = df.apply(lambda r: coord_to_h3(r['lat'], r['lon']), axis=1)

    # --- Extraer componentes temporales ---
    df['date']        = df['datetime_utc'].dt.date
    df['hour']        = df['datetime_utc'].dt.hour
    df['day_of_week'] = df['datetime_utc'].dt.dayofweek   # 0=lunes
    df['month']       = df['datetime_utc'].dt.month
    df['city']        = city

    return df[['city', 'h3_cell', 'datetime_utc', 'date', 'hour', 'day_of_week', 'month']]


print('Funciones definidas.')

## 3. Cargar y agregar todas las ciudades

In [ ]:
all_trips = []

for city in CITIES:
    print(f'\n[{city}]')
    df_city = load_city(city)
    print(f'  Viajes cargados: {len(df_city):,}')
    all_trips.append(df_city)

df_trips = pd.concat(all_trips, ignore_index=True)
print(f'\nTotal viajes: {len(df_trips):,}')
df_trips.head(3)

## 4. Agregación de demanda por (ciudad, h3_cell, fecha, hora)

In [ ]:
# Contar viajes iniciados por celda H3, día y hora
df_demand = (
    df_trips
    .groupby(['city', 'h3_cell', 'date', 'hour', 'day_of_week', 'month'])
    .size()
    .reset_index(name='target_demanda')
)

print(f'Filas con demanda > 0: {len(df_demand):,}')
print(f'Demanda media por celda-hora: {df_demand["target_demanda"].mean():.2f}')
df_demand.head()

## 5. Expansión con ceros — producto cartesiano completo

Se añaden las horas sin viajes (demanda = 0) para que el LSTM vea la serie temporal completa.

In [ ]:
city_frames = []

for city in CITIES:
    df_c = df_demand[df_demand['city'] == city].copy()
    if df_c.empty:
        continue

    # Todas las celdas activas y todas las fechas del período
    active_cells = df_c['h3_cell'].unique()
    all_dates    = pd.date_range(
        df_c['date'].min(), df_c['date'].max(), freq='D'
    ).date
    all_hours    = range(24)

    # Índice completo: celda × día × hora
    full_index = pd.DataFrame(
        list(product(active_cells, all_dates, all_hours)),
        columns=['h3_cell', 'date', 'hour']
    )
    full_index['city'] = city

    # Merge con demanda real; NaN → 0
    df_full = full_index.merge(
        df_c[['city', 'h3_cell', 'date', 'hour', 'day_of_week', 'month', 'target_demanda']],
        on=['city', 'h3_cell', 'date', 'hour'],
        how='left'
    )
    df_full['target_demanda'] = df_full['target_demanda'].fillna(0).astype(np.int16)

    # Rellenar day_of_week y month donde son NaN (horas sin viajes)
    df_full['date_dt']      = pd.to_datetime(df_full['date'])
    df_full['day_of_week']  = df_full['date_dt'].dt.dayofweek
    df_full['month']        = df_full['date_dt'].dt.month
    df_full.drop(columns='date_dt', inplace=True)

    city_frames.append(df_full)
    pct_zero = (df_full['target_demanda'] == 0).mean() * 100
    print(f'{city:12s} → {len(df_full):8,} filas | celdas activas: {len(active_cells):5,} | % ceros: {pct_zero:.1f}%')

dataset = pd.concat(city_frames, ignore_index=True)
print(f'\nDataset total: {len(dataset):,} filas')

## 6. Features temporales adicionales

In [ ]:
# Codificación cíclica de hora y día (útil para LSTM)
dataset['hour_sin'] = np.sin(2 * np.pi * dataset['hour'] / 24)
dataset['hour_cos'] = np.cos(2 * np.pi * dataset['hour'] / 24)
dataset['dow_sin']  = np.sin(2 * np.pi * dataset['day_of_week'] / 7)
dataset['dow_cos']  = np.cos(2 * np.pi * dataset['day_of_week'] / 7)

# Flag de fin de semana
dataset['is_weekend'] = (dataset['day_of_week'] >= 5).astype(np.int8)

print('Features cíclicas añadidas.')
dataset[['hour', 'hour_sin', 'hour_cos', 'day_of_week', 'dow_sin', 'dow_cos', 'is_weekend']].head(5)

## 7. Lags temporales por (ciudad, celda H3)

> ⚠️ Esta celda puede tardar varios minutos en datasets grandes. Los lags son críticos para el LSTM.

In [ ]:
# Ordenar cronológicamente dentro de cada grupo ciudad+celda
dataset.sort_values(['city', 'h3_cell', 'date', 'hour'], inplace=True)
dataset.reset_index(drop=True, inplace=True)

# Calcular lags por grupo (ciudad, celda)
g = dataset.groupby(['city', 'h3_cell'])['target_demanda']

dataset['lag_1h']          = g.shift(1)             # 1 hora antes
dataset['lag_24h']         = g.shift(24)            # mismo slot ayer
dataset['lag_168h']        = g.shift(168)           # mismo slot hace 1 semana
dataset['rolling_mean_3h'] = g.transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)                                                    # media 3h anteriores

# Estadísticas de NaN por lag (esperado: pocas primeras filas de cada grupo)
lag_cols = ['lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_3h']
print('NaN por columna de lag:')
print(dataset[lag_cols].isna().sum())

## 8. Resumen estadístico por ciudad

In [ ]:
summary = dataset.groupby('city').agg(
    filas          = ('target_demanda', 'count'),
    celdas_h3      = ('h3_cell', 'nunique'),
    demanda_media  = ('target_demanda', 'mean'),
    demanda_max    = ('target_demanda', 'max'),
    pct_ceros      = ('target_demanda', lambda x: (x == 0).mean() * 100),
    fecha_inicio   = ('date', 'min'),
    fecha_fin      = ('date', 'max'),
).round(2)

summary['dias_periodo'] = (
    pd.to_datetime(summary['fecha_fin']) - pd.to_datetime(summary['fecha_inicio'])
).dt.days + 1

print(summary.to_string())

## 9. Distribución de demanda por ciudad (por hora del día)

In [ ]:
import matplotlib.pyplot as plt

# Perfil horario medio de demanda (solo celdas con al menos un viaje)
df_nonzero = dataset[dataset['target_demanda'] > 0]
hourly_profile = df_nonzero.groupby(['city', 'hour'])['target_demanda'].mean().reset_index()

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=False)
axes = axes.flatten()

for i, city in enumerate(CITIES):
    df_plot = hourly_profile[hourly_profile['city'] == city]
    axes[i].bar(df_plot['hour'], df_plot['target_demanda'], color='steelblue', alpha=0.8)
    axes[i].set_title(city.capitalize())
    axes[i].set_xlabel('Hora')
    axes[i].set_ylabel('Demanda media')
    axes[i].set_xticks([0, 6, 12, 18, 23])

plt.suptitle('Perfil horario de demanda — celdas activas (demanda > 0)', fontsize=14)
plt.tight_layout()
plt.savefig('perfil_horario_ciudades.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Guardar dataset

In [ ]:
# Columnas finales del dataset
COLS_FINAL = [
    'city', 'h3_cell', 'date', 'hour',
    'day_of_week', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'target_demanda',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_3h'
]

dataset_final = dataset[COLS_FINAL].copy()

# Parquet: mucho más rápido de leer que CSV para datasets grandes
output_path = OUTPUT_DIR / 'dataset_h3_multicidad.parquet'
dataset_final.to_parquet(output_path, index=False)
print(f'Dataset guardado: {output_path}')
print(f'Tamaño: {output_path.stat().st_size / 1024**2:.1f} MB')
print(f'Shape: {dataset_final.shape}')
dataset_final.dtypes

## 11. Verificación rápida del dataset guardado

In [ ]:
# Carga de verificación
df_check = pd.read_parquet(OUTPUT_DIR / 'dataset_h3_multicidad.parquet')
print(f'Filas totales: {len(df_check):,}')
print(f'Ciudades: {sorted(df_check["city"].unique())}')
print(f'\nDistribución de demanda:')
print(df_check['target_demanda'].describe())
print(f'\nEjemplo de filas:')
df_check.sample(5)

---
## Notas para el siguiente paso (LSTM)

El dataset generado tiene la estructura correcta para construir secuencias LSTM:

- **Unidad de serie temporal:** cada `(city, h3_cell)` es una serie independiente de `T` pasos horarios
- **Features de entrada:** `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `is_weekend`, `lag_1h`, `lag_24h`, `lag_168h`, `rolling_mean_3h` + (opcional) variables socioeconómicas
- **Target:** `target_demanda` en el siguiente paso horario
- **Split ciudad de test:** excluir `muenchen` del preentrenamiento (período diferente: 2016 vs 2015)

```python
# Ejemplo de split
df_train = df_check[df_check['city'] != 'muenchen']
df_test  = df_check[df_check['city'] == 'muenchen']
```

**Resolución H3:** si hay demasiadas celdas con muy poca demanda (muchos ceros), probar resolución 7.